# Entrenamiento y comparación: kNN vs RandomForest

Mismo **split train/validación** que `scripts/train_fingerprint_model.py` (`random_state=123`, estrato por zona si hay suficientes muestras por clase).

En el camino: métricas train/val, **sweep de k** (kNN), **mapas de posición** y **matrices de confusión** en validación, **importancias** del RF (zona). Al final: **tabla y gráficos de barras** comparativos y **histograma** de la diferencia de error por muestra.


In [ ]:
%matplotlib inline
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config" / "baseline_room.yaml").is_file():
    REPO_ROOT = Path.cwd().parent
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from ble_indoor.evaluation.knn_sweep import sweep_k_neighbors
from ble_indoor.evaluation.metrics import position_errors_m
from ble_indoor.models.features import position_matrix, rssi_feature_matrix
from ble_indoor.models.fingerprint_knn import FingerprintKnnEstimator
from ble_indoor.models.fingerprint_rf import FingerprintRfEstimator
from ble_indoor.models.knn_zone import ZONE_ID_COLUMN
from ble_indoor.settings import ProjectConfig, ProjectLayout
from ble_indoor.simulation.omnet_trace_loader import load_omnet_training_trace

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)

layout = ProjectLayout(REPO_ROOT)
cfg = ProjectConfig.load(layout.default_config_path())
csv_path = layout.resolve_repo_path(cfg.omnet.training_trace_csv)
df = load_omnet_training_trace(csv_path, cfg.environment, cfg.spatial_zones)

y_zone = df[ZONE_ID_COLUMN].to_numpy(dtype=np.int64)
counts = np.bincount(y_zone, minlength=cfg.spatial_zones.n_zones)
strat = y_zone if int(np.min(counts)) >= 5 else None
train_df, val_df = train_test_split(
    df, test_size=0.25, random_state=123, shuffle=True, stratify=strat
)
env = cfg.environment
rssi_cols = [f"rssi_{g}" for g in env.gateway_ids]
zone_labels = cfg.spatial_zones.zone_labels()
label_ids = list(range(cfg.spatial_zones.n_zones))

print("CSV:", csv_path)
print("Train:", len(train_df), "| Val:", len(val_df))


## 1. kNN — ajuste y métricas


In [ ]:
knn = FingerprintKnnEstimator.from_config(cfg)
knn.fit(train_df, fit_zone=True)
m_knn_tr = knn.evaluate(train_df)
m_knn_va = knn.evaluate(val_df)

print("kNN — train:", m_knn_tr)
print("kNN — validation:", m_knn_va)


### Sweep de k (solo validación)

Analogía con `validation_vs_k.png` del script (`k_max=30`).


In [ ]:
K_MAX = 30
sweep = sweep_k_neighbors(train_df, val_df, env, cfg, k_max=K_MAX)
fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
ax0.plot(sweep["k"], sweep["validation_zone_accuracy"], "o-", color="tab:blue", markersize=4)
ax0.axvline(cfg.zone_knn.k_neighbors, color="gray", linestyle="--", label=f"YAML zone_knn.k={cfg.zone_knn.k_neighbors}")
ax0.set_ylabel("Zone accuracy (val)")
ax0.set_ylim(0, 1.02)
ax0.legend(fontsize=8)
ax0.set_title("kNN: validación vs k")
ax1.plot(sweep["k"], sweep["validation_rmse_xy_m"], "o-", color="tab:red", markersize=4)
ax1.set_xlabel("k")
ax1.set_ylabel("RMSE xy (m) val")
plt.tight_layout()
plt.show()


### Validación kNN: posición y confusión de zona


In [ ]:
def draw_room_positions(ax, true_xy, pred_xy, title: str) -> None:
    ax.set_aspect("equal", adjustable="box")
    ax.add_patch(
        plt.Rectangle((0, 0), env.room.width_m, env.room.height_m, fill=False, edgecolor="black", linewidth=1.2)
    )
    gw = env.gateway_positions_m()
    ax.scatter(gw[:, 0], gw[:, 1], c="tab:blue", s=100, zorder=3, label="GW")
    ax.scatter(train_df["x_m"], train_df["y_m"], c="lightgray", s=10, alpha=0.4, label="train")
    ax.scatter(true_xy[:, 0], true_xy[:, 1], c="tab:green", s=22, label="truth", zorder=2)
    ax.scatter(pred_xy[:, 0], pred_xy[:, 1], c="tab:red", s=16, marker="x", label="est", zorder=2)
    for i in range(len(true_xy)):
        ax.plot([true_xy[i, 0], pred_xy[i, 0]], [true_xy[i, 1], pred_xy[i, 1]], c="tab:orange", alpha=0.25, linewidth=0.8)
    ax.set_xlim(-0.5, env.room.width_m + 0.5)
    ax.set_ylim(-0.5, env.room.height_m + 0.5)
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_title(title)
    ax.legend(loc="upper right", fontsize=7)


Xv = rssi_feature_matrix(val_df, env)
true_xy = position_matrix(val_df)
pred_xy_knn = knn.predict_xy_batch(Xv)
err_knn = position_errors_m(true_xy, pred_xy_knn)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
draw_room_positions(axes[0], true_xy, pred_xy_knn, "kNN — validación")
im = axes[1].hexbin(true_xy[:, 0], true_xy[:, 1], C=err_knn, gridsize=22, cmap="magma", reduce_C_function=np.mean)
axes[1].set_aspect("equal")
axes[1].set_xlabel("x true (m)")
axes[1].set_ylabel("y true (m)")
axes[1].set_title("kNN — error (m) medio por celda")
plt.colorbar(im, ax=axes[1], label="error (m)")
plt.tight_layout()
plt.show()

z_t = val_df[ZONE_ID_COLUMN].to_numpy(dtype=np.int64)
z_p_knn = knn.predict_zone_batch(Xv)
cm_knn = confusion_matrix(z_t, z_p_knn, labels=label_ids)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm_knn, annot=True, fmt="d", cmap="Blues", xticklabels=zone_labels, yticklabels=zone_labels, ax=ax)
ax.set_xlabel("Predicho")
ax.set_ylabel("Verdadero")
ax.set_title("kNN — confusión zona (validación)")
plt.tight_layout()
plt.show()


## 2. RandomForest — ajuste y métricas


In [ ]:
rf = FingerprintRfEstimator.from_config(cfg)
rf.fit(train_df, fit_zone=True)
m_rf_tr = rf.evaluate(train_df)
m_rf_va = rf.evaluate(val_df)

print("RF — train:", m_rf_tr)
print("RF — validation:", m_rf_va)


### Importancia de RSSI en el clasificador de zona (RF)


In [ ]:
zone_rf = rf._zone
if zone_rf is not None:
    imp = zone_rf.feature_importances_
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.barh(rssi_cols, imp, color="seagreen")
    ax.set_xlabel("importancia")
    ax.set_title("RandomForestClassifier (zona): feature_importances_")
    plt.tight_layout()
    plt.show()


### Validación RF: posición y confusión


In [ ]:
pred_xy_rf = rf.predict_xy_batch(Xv)

fig, ax = plt.subplots(figsize=(6, 5.5))
draw_room_positions(ax, true_xy, pred_xy_rf, "RandomForest — validación")
plt.tight_layout()
plt.show()

z_p_rf = rf.predict_zone_batch(Xv)
cm_rf = confusion_matrix(z_t, z_p_rf, labels=label_ids)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Greens", xticklabels=zone_labels, yticklabels=zone_labels, ax=ax)
ax.set_xlabel("Predicho")
ax.set_ylabel("Verdadero")
ax.set_title("RF — confusión zona (validación)")
plt.tight_layout()
plt.show()


## 3. Comparación de métricas


In [ ]:
def flat_metrics(prefix: str, m_tr: dict, m_va: dict) -> dict:
    pos_tr, pos_va = m_tr["position"], m_va["position"]
    row = {
        "modelo": prefix,
        "train_rmse_xy_m": pos_tr["rmse_xy_m"],
        "val_rmse_xy_m": pos_va["rmse_xy_m"],
        "train_r2": pos_tr["r2"],
        "val_r2": pos_va["r2"],
    }
    if "zone" in m_tr and "zone" in m_va:
        row["train_zone_acc"] = m_tr["zone"]["accuracy"]
        row["val_zone_acc"] = m_va["zone"]["accuracy"]
    return row

cmp = pd.DataFrame([
    flat_metrics("kNN", m_knn_tr, m_knn_va),
    flat_metrics("RandomForest", m_rf_tr, m_rf_va),
])
try:
    from IPython.display import display
    display(cmp)
except Exception:
    print(cmp.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
x = np.arange(2)
w = 0.35
axes[0].bar(x - w / 2, cmp["val_rmse_xy_m"], w, label="val", color="coral")
axes[0].bar(x + w / 2, cmp["train_rmse_xy_m"], w, label="train", color="steelblue")
axes[0].set_xticks(x)
axes[0].set_xticklabels(cmp["modelo"])
axes[0].set_ylabel("RMSE xy (m)")
axes[0].set_title("Error de posición")
axes[0].legend()

axes[1].bar(x - w / 2, cmp["val_r2"], w, label="val", color="coral")
axes[1].bar(x + w / 2, cmp["train_r2"], w, label="train", color="steelblue")
axes[1].set_xticks(x)
axes[1].set_xticklabels(cmp["modelo"])
axes[1].set_ylabel("R²")
axes[1].set_title("R² (posición)")
axes[1].legend()

axes[2].bar(x - w / 2, cmp["val_zone_acc"], w, label="val", color="coral")
axes[2].bar(x + w / 2, cmp["train_zone_acc"], w, label="train", color="steelblue")
axes[2].set_xticks(x)
axes[2].set_xticklabels(cmp["modelo"])
axes[2].set_ylabel("accuracy")
axes[2].set_ylim(0, 1.05)
axes[2].set_title("Zona")
axes[2].legend()

plt.suptitle("kNN vs RandomForest (mismo split)", y=1.02)
plt.tight_layout()
plt.show()


### Error de posición por muestra: kNN − RF

**> 0** → RF mejor en esa muestra; **< 0** → kNN mejor.


In [ ]:
err_rf = position_errors_m(true_xy, pred_xy_rf)
delta = err_knn - err_rf
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(delta, bins=50, color="purple", alpha=0.75, edgecolor="none")
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("error_knn - error_rf (m)")
ax.set_ylabel("conteos")
ax.set_title("Validación: comparación local de error")
plt.tight_layout()
plt.show()
print("Media (knn - rf):", float(np.mean(delta)), "m | fracción RF mejor:", float(np.mean(delta > 0)))
